# **5. Evaluation & Plots**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive, files

In [ ]:
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/commonsenseqa_parpahrase_evaluation"

eval_df = pd.read_csv(f"{BASE_PATH}/eval_df.csv")
results_exp1 = pd.read_csv(f"{BASE_PATH}/results_exp1.csv")
results_exp1_semantic = pd.read_csv(f"{BASE_PATH}/results_exp1_semantic.csv")
results_exp2 = pd.read_csv(f"{BASE_PATH}/results_exp2.csv")
results_exp2_semantic = pd.read_csv(f"{BASE_PATH}/results_exp2_semantic.csv")

In [ ]:
def save_df(df, filename):
    path = os.path.join(BASE_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

def save_plot(filename):
    path = os.path.join(BASE_PATH, filename)
    plt.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Saved plot: {path}")

### Add Consistency

In [ ]:
def add_consistency(results_df):
    results_df = results_df.copy()

    # remove old columns if they already exist from a previous run
    drop_cols = ["original_prediction", "consistent_with_original"]
    results_df = results_df.drop(columns=[c for c in drop_cols if c in results_df.columns])

    # get each model's prediction on the original question
    original_preds = (
        results_df.loc[results_df["variant_name"] == "original", ["model", "id", "predicted_answer"]]
        .rename(columns={"predicted_answer": "original_prediction"})
        .drop_duplicates(subset=["model", "id"])
    )

    # merge original prediction back to all variants of the same question
    merged = results_df.merge(original_preds, on=["model", "id"], how="left")

    # compute consistency
    merged["consistent_with_original"] = (
        merged["predicted_answer"].notna() &
        merged["original_prediction"].notna() &
        (merged["predicted_answer"] == merged["original_prediction"])
    )

    # originals should always be consistent with themselves
    merged.loc[merged["variant_name"] == "original", "consistent_with_original"] = True

    return merged

In [ ]:
results_exp1 = add_consistency(results_exp1)
results_exp1_semantic = add_consistency(results_exp1_semantic)
results_exp2 = add_consistency(results_exp2)
results_exp2_semantic = add_consistency(results_exp2_semantic)

save_df_to_drive(results_exp1, "results_exp1.csv")
save_df_to_drive(results_exp1_semantic, "results_exp1_semantic.csv")
save_df_to_drive(results_exp2, "results_exp2.csv")
save_df_to_drive(results_exp2_semantic, "results_exp2_semantic.csv")

### Result Summaries

In [ ]:
summary_exp1 = results_exp1.groupby(["model", "paraphrase_level"]).agg(
    n=("id","count"),
    accuracy=("correct","mean"),
    consistency=("consistent_with_original","mean")
).reset_index()

summary_exp1
save_df_to_drive(summary_exp1, "summary_exp1.csv")

In [ ]:
summary_exp1_semantic = results_exp1_semantic.groupby(["model", "paraphrase_level"]).agg(
    n=("id","count"),
    accuracy=("correct","mean"),
    consistency=("consistent_with_original","mean")
).reset_index()

summary_exp1_semantic
save_df_to_drive(summary_exp1_semantic, "summary_exp1_semantic.csv")

In [ ]:
summary_exp2 = results_exp2.groupby(["model", "paraphrase_level"]).agg(
    n=("id","count"),
    accuracy=("correct","mean"),
    consistency=("consistent_with_original","mean")
).reset_index()

summary_exp2
save_df_to_drive(summary_exp2, "summary_exp2.csv")

In [ ]:
summary_exp2_semantic = results_exp2_semantic.groupby(["model", "paraphrase_level"]).agg(
    n=("id","count"),
    accuracy=("correct","mean"),
    consistency=("consistent_with_original","mean")
).reset_index()

summary_exp2_semantic
save_df_to_drive(summary_exp2_semantic, "summary_exp2_semantic.csv")

### Accuracy & Consistency Plots

In [ ]:
# Combine summaries
plot_all = pd.concat([summary_exp1, summary_exp2], ignore_index=True)
plot_sem = pd.concat([summary_exp1_semantic, summary_exp2_semantic], ignore_index=True)

# Remove duplicates
plot_all = plot_all.drop_duplicates(subset=["model", "paraphrase_level"])
plot_sem = plot_sem.drop_duplicates(subset=["model", "paraphrase_level"])

# Order levels
level_order = ["original", "light", "medium", "strong"]

for df in [plot_all, plot_sem]:
    df["paraphrase_level"] = pd.Categorical(
        df["paraphrase_level"],
        categories=level_order,
        ordered=True
    )

plot_all = plot_all.sort_values(["model", "paraphrase_level"])
plot_sem = plot_sem.sort_values(["model", "paraphrase_level"])

# Create side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14,5), sharey=True)

# --- Plot 1: No semantic filtering ---
for model_name in plot_all["model"].unique():
    temp = plot_all[plot_all["model"] == model_name]
    axes[0].plot(temp["paraphrase_level"], temp["accuracy"], marker="o", label=model_name)

axes[0].set_title("Accuracy (All Paraphrases)")
axes[0].set_xlabel("Paraphrase Level")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Semantic filtering ---
for model_name in plot_sem["model"].unique():
    temp = plot_sem[plot_sem["model"] == model_name]
    axes[1].plot(temp["paraphrase_level"], temp["accuracy"], marker="o", label=model_name)

axes[1].set_title("Accuracy (Semantic-Filtered)")
axes[1].set_xlabel("Paraphrase Level")
axes[1].grid(True, alpha=0.3)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels))

plt.tight_layout(rect=[0,0,1,0.92])
plt.show()

In [ ]:
# Combine summaries
plot_all = pd.concat([summary_exp1, summary_exp2], ignore_index=True)
plot_sem = pd.concat([summary_exp1_semantic, summary_exp2_semantic], ignore_index=True)

# Remove duplicates
plot_all = plot_all.drop_duplicates(subset=["model", "paraphrase_level"])
plot_sem = plot_sem.drop_duplicates(subset=["model", "paraphrase_level"])

# Order levels
level_order = ["original", "light", "medium", "strong"]

for df in [plot_all, plot_sem]:
    df["paraphrase_level"] = pd.Categorical(
        df["paraphrase_level"],
        categories=level_order,
        ordered=True
    )

plot_all = plot_all.sort_values(["model", "paraphrase_level"])
plot_sem = plot_sem.sort_values(["model", "paraphrase_level"])

# Create side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14,5), sharey=True)

# --- Plot 1: No semantic filtering ---
for model_name in plot_all["model"].unique():
    temp = plot_all[plot_all["model"] == model_name]
    axes[0].plot(temp["paraphrase_level"], temp["consistency"], marker="o", label=model_name)

axes[0].set_title("Consistency (All Paraphrases)")
axes[0].set_xlabel("Paraphrase Level")
axes[0].set_ylabel("Consistency with Original")
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Semantic filtering ---
for model_name in plot_sem["model"].unique():
    temp = plot_sem[plot_sem["model"] == model_name]
    axes[1].plot(temp["paraphrase_level"], temp["consistency"], marker="o", label=model_name)

axes[1].set_title("Consistency (Semantic-Filtered)")
axes[1].set_xlabel("Paraphrase Level")
axes[1].grid(True, alpha=0.3)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels))

plt.tight_layout(rect=[0,0,1,0.92])
plt.show()

### NLI Scoring

In [ ]:
# Keep only paraphrased rows for semantic analysis
nli_plot_df = eval_df[eval_df["paraphrase_level"].isin(["light", "medium", "strong"])].copy()

heatmap_df = nli_plot_df.groupby("paraphrase_level")[[
    "nli_entailment", "nli_neutral", "nli_contradiction"
]].mean()

heatmap_df = heatmap_df.loc[["light", "medium", "strong"]]

fig, ax = plt.subplots(figsize=(8, 4.5))

im = ax.imshow(heatmap_df.values, aspect="auto", cmap="viridis")

ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(["Entailment", "Neutral", "Contradiction"], fontsize=11)

ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels([lvl.title() for lvl in heatmap_df.index], fontsize=11)

ax.set_title("Average NLI Scores by Paraphrase Level", fontsize=13, pad=12)

# Add readable values inside cells
for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        value = heatmap_df.values[i, j]
        text_color = "white" if value < 0.6 else "black"
        ax.text(
            j, i, f"{value:.2f}",
            ha="center", va="center",
            color=text_color, fontsize=11, fontweight="bold"
        )

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Average score", rotation=270, labelpad=15)

plt.tight_layout()
save_plot_to_drive("nli_heatmap_readable.png")
plt.show()

save_df_to_drive(heatmap_df.reset_index(), "nli_heatmap_values.csv")

### Weighted & Variance Prompt Sensitivity

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Use the semantic-filtered results already created in this notebook
df1 = results_exp1_semantic.copy()
df2 = results_exp2_semantic.copy()

df1["source"] = "exp1"
df2["source"] = "exp2"

df = pd.concat([df1, df2], ignore_index=True)

# Cleaning
df["paraphrase_level"] = df["paraphrase_level"].astype(str).str.strip().str.lower()
df["model"] = df["model"].astype(str).str.strip()

# Remove original rows for sensitivity metrics
df_para = df[df["paraphrase_level"] != "original"].copy()

# Keep one value per model × level for consistency
consistency = (
    df_para.groupby(["model", "paraphrase_level"])["consistent_with_original"]
    .mean()
    .unstack()
    .reindex(columns=["light", "medium", "strong"])
)

# Define weights
w_light = 0.2
w_medium = 0.3
w_strong = 0.5

# Weighted Prompt Sensitivity Index
consistency["W_PSI"] = (
    w_light * (1 - consistency["light"]) +
    w_medium * (1 - consistency["medium"]) +
    w_strong * (1 - consistency["strong"])
)

wpsi_df = consistency.reset_index().round(3)
wpsi_df
save_df_to_drive(wpsi_df, "wpsi_summary.csv")

# Answer Variance Sensitivity (AVS)
def compute_variance(group):
    probs = group["predicted_answer"].value_counts(normalize=True)
    return 1 - (probs ** 2).sum()

avs_df = (
    df_para.groupby(["model", "id"])
    .apply(compute_variance)
    .groupby("model")
    .mean()
    .reset_index(name="AVS")
    .round(3)
)

avs_df
save_df_to_drive(avs_df, "avs_summary.csv")

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(wpsi_df["model"], wpsi_df["W_PSI"])
plt.title("Weighted Prompt Sensitivity Index (W-PSI)")
plt.xlabel("Model")
plt.ylabel("Weighted Sensitivity")
plt.ylim(0, 1)
plt.tight_layout()
save_plot_to_drive("wpsi_plot.png")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(avs_df["model"], avs_df["AVS"])
plt.title("Average Variance Sensitivity (AVS)")
plt.xlabel("Model")
plt.ylabel("Variance Sensitivity")
plt.ylim(0, 1)
plt.tight_layout()
save_plot_to_drive("avs_plot.png")
plt.show()

### Failure Taxonomy

In [ ]:
import pandas as pd

LEVELS = ["light", "medium", "strong"]
CAT_ORDER = [
    "Stable Correct",
    "Brittle Failure",
    "Unstable but Recoverable",
    "Stable Wrong",
]
CATEGORIES = {
    (True,  True ): "Stable Correct",
    (True,  False): "Brittle Failure",
    (False, True ): "Unstable but Recoverable",
    (False, False): "Stable Wrong",
}

# Build one combined results table with original + all paraphrase levels
taxonomy_source = pd.concat([
    results_exp1[results_exp1["paraphrase_level"].isin(["original", "light"])],
    results_exp2[results_exp2["paraphrase_level"].isin(["original", "medium", "strong"])],
], ignore_index=True)

taxonomy_source = taxonomy_source.drop_duplicates(
    subset=["model", "id", "variant_name", "paraphrase_level"]
).copy()

taxonomy_source["correct"] = taxonomy_source["correct"].astype(bool)

orig_df = taxonomy_source[taxonomy_source["variant_name"] == "original"][
    ["model", "id", "gold_answer", "predicted_answer", "correct"]
].drop_duplicates(subset=["model", "id"]).rename(columns={
    "answerKey": "gold_answer",
    "predicted_answer": "orig_pred",
    "correct": "orig_correct",
})

para = taxonomy_source[taxonomy_source["paraphrase_level"].isin(LEVELS)][
    ["model", "id", "variant_name", "paraphrase_level",
     "question_text", "predicted_answer", "correct"]
].copy().rename(columns={
    "predicted_answer": "para_pred",
    "correct": "para_correct"
})

print(f"Original rows available: {len(orig_df):,}")
print(f"Paraphrase rows available: {len(para):,}")

In [ ]:
# Per individual variant
var_df = para.merge(
    orig_df[["model", "id", "gold_answer", "orig_pred", "orig_correct"]],
    on=["model", "id"],
    how="left",
)

var_df["taxonomy"] = var_df.apply(
    lambda r: CATEGORIES[(bool(r["orig_correct"]), bool(r["para_correct"]))], axis=1
)

save_df_to_drive(var_df, "taxonomy_per_variant.csv")

# Per paraphrase level (majority vote across variants in each level)
level_agg = (
    para.groupby(["model", "id", "paraphrase_level"])["para_correct"]
    .agg(variants_correct="sum", variants_total="count")
    .reset_index()
)
level_agg["para_majority_correct"] = (
    level_agg["variants_correct"] > level_agg["variants_total"] / 2
)

level_df = level_agg.merge(
    orig_df[["model", "id", "gold_answer", "orig_pred", "orig_correct"]],
    on=["model", "id"],
    how="left",
)

level_df["taxonomy"] = level_df.apply(
    lambda r: CATEGORIES[(bool(r["orig_correct"]), bool(r["para_majority_correct"]))], axis=1
)

save_df_to_drive(level_df, "taxonomy_per_level.csv")

In [ ]:
def make_pivot(df_in, group_cols):
    counts = (
        df_in.groupby(group_cols + ["taxonomy"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=CAT_ORDER, fill_value=0)
    )
    counts["Total"] = counts.sum(axis=1)
    pct = counts[CAT_ORDER].div(counts["Total"], axis=0).mul(100).round(1)
    pct.columns = [f"{c} %" for c in pct.columns]
    return pd.concat([counts, pct], axis=1)

taxonomy_model_level_variant = make_pivot(var_df, ["model", "paraphrase_level"])
taxonomy_model_level_majority = make_pivot(level_df, ["model", "paraphrase_level"])
taxonomy_by_model = make_pivot(var_df, ["model"])
taxonomy_by_level = make_pivot(var_df, ["paraphrase_level"]).reindex(LEVELS, fill_value=0)

display(taxonomy_model_level_variant)
display(taxonomy_model_level_majority)
display(taxonomy_by_model)
display(taxonomy_by_level)

save_df_to_drive(taxonomy_model_level_variant.reset_index(), "taxonomy_model_level_variant_summary.csv")
save_df_to_drive(taxonomy_model_level_majority.reset_index(), "taxonomy_model_level_majority_summary.csv")
save_df_to_drive(taxonomy_by_model.reset_index(), "taxonomy_by_model_summary.csv")
save_df_to_drive(taxonomy_by_level.reset_index(), "taxonomy_by_level_summary.csv")

with pd.ExcelWriter(PROJECT_DIR / "taxonomy_summary.xlsx") as writer:
    taxonomy_model_level_variant.to_excel(writer, sheet_name="Model x Level (variant)")
    taxonomy_model_level_majority.to_excel(writer, sheet_name="Model x Level (majority)")
    taxonomy_by_model.to_excel(writer, sheet_name="By Model")
    taxonomy_by_level.to_excel(writer, sheet_name="By Level")

print(f"Saved -> {PROJECT_DIR / 'taxonomy_summary.xlsx'}")

### High Risk Analysis

In [ ]:
print("Loaded shapes:")
for name, df in [("exp1", results_exp1), ("exp2", results_exp2),
                  ("exp1_semantic", results_exp1_semantic), ("exp2_semantic", results_exp2_semantic)]:
    print(f"  {name}: {df.shape}")

# Combine all results, drop duplicates
all_results = pd.concat([results_exp1, results_exp2], ignore_index=True)
all_results = all_results.drop_duplicates(subset=["model", "id", "variant_name"])

all_semantic = pd.concat([results_exp1_semantic, results_exp2_semantic], ignore_index=True)
all_semantic = all_semantic.drop_duplicates(subset=["model", "id", "variant_name"])

# --- Reconstruct semantic_ok=False via set difference ---
semantic_ok_keys = set(
    zip(all_semantic["id"], all_semantic["variant_name"])
)

all_results["semantic_ok"] = [
    (i, v) in semantic_ok_keys
    for i, v in zip(all_results["id"], all_results["variant_name"])
]

# Originals are always semantic_ok=True
all_results.loc[all_results["variant_name"] == "original", "semantic_ok"] = True

print(f"\nsemantic_ok breakdown:")
print(all_results[all_results["variant_name"] != "original"]["semantic_ok"].value_counts())

In [ ]:
# Keep only semantic_ok=False paraphrase rows
failed_results = all_results[
    (all_results["variant_name"] != "original") &
    (all_results["semantic_ok"] == False)
].copy()

print(f"semantic_ok=False paraphrase rows: {len(failed_results)}")
print(f"Unique questions: {failed_results['id'].nunique()}")

# For each (id, variant_name): did ANY model flip on this variant?
per_variant = failed_results.groupby(["id", "variant_name"]).agg(
    any_flip        =("consistent_with_original", lambda x: (~x).any()),
    n_models_flipped=("consistent_with_original", lambda x: (~x).sum()),
).reset_index()

# correct-wrong: all models correct on original AND any model wrong on this variant
orig_all_correct = (
    all_results[all_results["variant_name"] == "original"]
    .groupby("id")["correct"]
    .all()
    .reset_index()
    .rename(columns={"correct": "orig_all_correct"})
)
any_wrong_on_para = (
    failed_results.groupby(["id", "variant_name"])["correct"]
    .apply(lambda x: (~x).any())
    .reset_index()
    .rename(columns={"correct": "any_wrong_on_para"})
)
per_variant = per_variant.merge(orig_all_correct, on="id", how="left")
per_variant = per_variant.merge(any_wrong_on_para, on=["id", "variant_name"], how="left")
per_variant["correct_to_wrong"] = per_variant["orig_all_correct"] & per_variant["any_wrong_on_para"]

# Roll up to question level
q_stats = per_variant.groupby("id").agg(
    n_failed_paraphrases  =("variant_name",    "count"),
    any_model_flip_rate   =("any_flip",         "mean"),
    correct_to_wrong_rate =("correct_to_wrong", "mean"),
    mean_n_models_flipped =("n_models_flipped", "mean"),
).reset_index()

q_stats = q_stats.sort_values(
    ["any_model_flip_rate", "correct_to_wrong_rate"],
    ascending=False
).reset_index(drop=True)

print(f"\nTop 10 most unstable questions:")
q_stats.head(10)


In [ ]:
TOP_N = 100
top100_ids = q_stats.head(TOP_N)["id"].tolist()

# use original variant rows
original_text = (
    all_results[all_results["variant_name"] == "original"][["id", "question_text"]]
    .drop_duplicates(subset="id")
    .rename(columns={"question_text": "original"})
)

# Pivot all paraphrase variant texts wide for the top 100 questions
para_text = (
    all_results[
        (all_results["id"].isin(top100_ids)) &
        (all_results["variant_name"] != "original")
    ][["id", "variant_name", "question_text"]]
    .drop_duplicates(subset=["id", "variant_name"])
)
text_pivot = para_text.pivot_table(
    index="id", columns="variant_name", values="question_text", aggfunc="first"
).reset_index()
level_cols = [c for c in text_pivot.columns if c.startswith(("light", "medium", "strong"))]
text_pivot = text_pivot[["id"] + sorted(level_cols)]

# Which variants were semantic_ok=False AND caused any model to flip
failed_variant_list = (
    per_variant[per_variant["id"].isin(top100_ids)]
    .groupby("id")["variant_name"]
    .apply(lambda x: ", ".join(sorted(x)))
    .reset_index().rename(columns={"variant_name": "failed_semantic_variants"})
)
flipped_variant_list = (
    per_variant[per_variant["id"].isin(top100_ids) & per_variant["any_flip"]]
    .groupby("id")["variant_name"]
    .apply(lambda x: ", ".join(sorted(x)))
    .reset_index().rename(columns={"variant_name": "flipped_variants"})
)

# Assemble final review table
review_df = (
    original_text[original_text["id"].isin(top100_ids)]
    .merge(text_pivot, on="id", how="left")
    .merge(q_stats[["id", "n_failed_paraphrases", "any_model_flip_rate",
                    "correct_to_wrong_rate", "mean_n_models_flipped"]], on="id", how="left")
    .merge(failed_variant_list,  on="id", how="left")
    .merge(flipped_variant_list, on="id", how="left")
    .sort_values("any_model_flip_rate", ascending=False)
    .reset_index(drop=True)
)

print(f"Review table: {len(review_df)} questions x {review_df.shape[1]} columns")
review_df[["id", "original", "n_failed_paraphrases",
           "any_model_flip_rate", "correct_to_wrong_rate",
           "failed_semantic_variants", "flipped_variants"]].head(5)


In [ ]:
# --- Split into 5 chunks of 20 for each group member ---
N_REVIEWERS    = 5
CHUNK_SIZE     = TOP_N // N_REVIEWERS  # 20
REVIEWER_NAMES = ["reviewer_1", "reviewer_2", "reviewer_3", "reviewer_4", "reviewer_5"]

review_chunks = {}
for i, name in enumerate(REVIEWER_NAMES):
    chunk = review_df.iloc[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE].copy()


    chunk["failure_type"] = ""  # synonym_change | negation | word_order | ambiguity | commonsense | other
    chunk["notes"]        = ""  # free text observations

    review_chunks[name] = chunk
    print(f"{name}: {len(chunk)} questions  "
          f"(flip rate {chunk['any_model_flip_rate'].min():.2f}–{chunk['any_model_flip_rate'].max():.2f}, "
          f"correct-wrong {chunk['correct_to_wrong_rate'].mean():.2f})")

print()
print("failure_type options: synonym_change | negation | word_order | ambiguity | commonsense | other")


In [ ]:
os.makedirs("review_chunks", exist_ok=True)

for name, chunk in review_chunks.items():
    path = f"review_chunks/high_risk_{name}.csv"
    chunk.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(chunk)} rows)")

In [ ]:
for name in REVIEWER_NAMES:
    files.download(f"review_chunks/high_risk_{name}.csv")